# 02 — Prompt Engineering for Gemini Grounding

This notebook documents the design rationale behind the prompts used in `api/gemini_client.py`, showing 2-3 iterations toward the final grounded prompt, the hallucination risks considered at each step, and the mitigations applied.

**Important**: `GEMINI_API_KEY` is **not set** in this build environment (by design — this is a zero-cost project and the AI layer is entirely optional). So this notebook does **not** execute live Gemini API calls; it documents the prompt design process and shows the exact final prompt text that `gemini_client.py` sends, ready to run the moment a free key from [aistudio.google.com](https://aistudio.google.com) is added to the environment.

Disclaimer: Educational project. Not investment advice.

## The core risk: hallucinated financial figures

The single biggest risk of putting an LLM in front of financial data is that it will state a plausible-sounding but **fabricated** number (a made-up P/E ratio, an invented growth rate, a guessed price target) with the same confident tone as a real, sourced figure. For a portfolio project explicitly framed as "educational, not investment advice," this is unacceptable even in a low-stakes demo — so the entire prompt design process below is oriented around eliminating that risk as thoroughly as possible without a fact-checking/RAG pipeline (which would be overkill and add cost/complexity for a zero-cost project).

## Iteration 1 — Naive prompt (rejected)

The first prompt draft considered was simple and unconstrained:

```text
Write a research note about {ticker}.
```

**Why this was rejected**: With no data provided at all, the model has only two options: refuse, or hallucinate from its training data (which may be stale, wrong, or simply doesn't know a smaller/less-covered ticker). Given `gemini-2.5-flash`'s training cutoff is unrelated to the live data this app computes, any specific number it produced (a P/E, a price, a margin) would be untrustworthy by construction. This iteration made clear that **the prompt must carry the actual computed data itself** rather than relying on the model's own knowledge.

## Iteration 2 — Data included, but grounding instruction too soft (rejected)

The second draft included the computed metrics but only a mild grounding hint:

```text
Here is some data about {ticker}: {data}.
Please write a research note. Try to be accurate.
```

**Why this was rejected**: "Try to be accurate" doesn't tell the model what to do when a field is missing (e.g. `forward_pe: null`) — it could easily fill the gap with a plausible-sounding estimate instead of saying "not available." It also doesn't explicitly forbid using outside/training-data knowledge to *supplement* the provided numbers (e.g. "Apple's P/E has historically been around X" — a real risk for well-known, heavily-covered tickers like AAPL where the model likely does have memorized figures that could leak in and contradict, or worse, subtly blend with, the actual computed numbers).

## Iteration 3 — Final: explicit system instruction + structured JSON context + per-call reminder (adopted)

The final design, implemented in `gemini_client.py`, layers grounding at **three levels**:

1. **System instruction** (set once when the model is initialized via `genai.GenerativeModel(system_instruction=...)`) — a durable behavioral constraint:

   > You are an equity research assistant embedded in a portfolio project. Only use the numbers provided below. Never invent or estimate financial figures not given to you. If information is missing, say so explicitly. Always keep a neutral, educational tone. Never phrase output as personalized investment advice.

2. **Structured JSON data block**, generated directly from the same `analytics.py` output rendered in the UI (so there is zero opportunity for a data mismatch between what the user sees on screen and what the model is given):

   ```json
   {
     "ticker": "AAPL",
     "metrics": {"last_close": 313.39, "sma_50": 295.85, "sma_200": 271.44, "rsi_14": 59.49, "...": "..."},
     "fundamentals": {"trailing_pe": 37.94, "market_cap": 4602870628352, "...": "..."},
     "peers": []
   }
   ```

3. **Per-call task instruction that repeats the constraint**, so grounding survives even in edge cases where a client library might not consistently apply the system instruction:

   > Using ONLY the structured data below for ticker {ticker}, write a research note with these exact section headers... If a needed number is missing/null, explicitly say so instead of guessing.

This redundancy (system instruction + explicit data-only framing + per-call reminder) is intentional defense-in-depth, not duplication for its own sake.

## Final prompt text (verbatim, ready to run)

The cells below reproduce the **exact** prompt-construction logic from `api/gemini_client.py`, so this notebook is directly runnable once `GEMINI_API_KEY` is available (just uncomment the live call at the bottom).

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import json
from api import gemini_client as gc

print("GEMINI_API_KEY set?", bool(os.environ.get("GEMINI_API_KEY")))
print("Model name:", gc.MODEL_NAME)
print()
print("System instruction (sent once per model init):")
print(gc.SYSTEM_INSTRUCTION)

Expected output when `GEMINI_API_KEY` is not set (as in this build environment) — this is the **actual, real output** of running the cell above in this exact environment:

```
GEMINI_API_KEY set? False
Model name: gemini-2.5-flash

System instruction (sent once per model init):
You are an equity research assistant embedded in a portfolio project. Only use the numbers provided below. Never invent or estimate financial figures not given to you. If information is missing, say so explicitly. Always keep a neutral, educational tone. Never phrase output as personalized investment advice.
```

In [ ]:
sample_metrics = {
    "last_close": 313.39,
    "sma_50": 295.85,
    "sma_200": 271.44,
    "golden_cross": True,
    "rsi_14": 59.49,
    "return_1m": 0.0197,
    "return_6m": 0.1748,
}
sample_fundamentals = {
    "trailing_pe": 37.94,
    "market_cap": 4602870628352,
    "profit_margin": 0.2715,
    "roe": 1.4147,
    "short_name": "Apple Inc.",
    "sector": "Technology",
    "industry": "Consumer Electronics",
}
sample_peers = [{"symbol": "MSFT", "trailing_pe": 22.82}, {"symbol": "GOOGL", "trailing_pe": 27.63}]

context_block = gc._serialize_context("AAPL", sample_metrics, sample_fundamentals, sample_peers)

final_prompt = (
    f"Using ONLY the structured data below for ticker AAPL, write a research note with these "
    f"exact section headers: '## Business Snapshot', '## Valuation View', '## Momentum/Technicals', "
    f"'## Risks', '## Questions for Further Research'. Be concise (2-4 sentences per section). "
    f"If a needed number is missing/null, explicitly say so instead of guessing.\n\n"
    f"DATA:\n{context_block}"
)
print(final_prompt)

This is the literal, byte-for-byte prompt text `gemini_client.generate_research_note()` sends to `model.generate_content(prompt)` when a `GEMINI_API_KEY` is configured. Note the JSON `DATA` block is generated by `_serialize_context`, the same function used for the chat and peer-explanation prompts — it is the single point of truth for what the model is allowed to "know."

In [ ]:
# Live call — only runs meaningfully if GEMINI_API_KEY is set in the environment.
# With no key set (as in this build environment), generate_research_note() falls back
# to the rule-based templated summary instead of raising, which we verify below.

result = gc.generate_research_note("AAPL", metrics=sample_metrics, fundamentals=sample_fundamentals, peers=sample_peers)
print(result["text"][:600])

Actual output observed when running this cell in the build environment (no `GEMINI_API_KEY` set):

```
This is a rule-based summary -- set GEMINI_API_KEY for AI-generated analysis.

## Business Snapshot
Apple Inc. (AAPL) -- sector: Technology, industry: Consumer Electronics. Market cap: 4602870628352.00.

## Valuation View
P/E of 37.94 is above the peer average of 25.23.

## Momentum / Technicals
Price is currently above both its 50-day and 200-day moving averages (bullish trend). RSI(14) is 59.5, which is in a neutral range.

## Risks
...
```

This confirms the fallback path produces the same section structure the AI-generated version would, using only the data actually provided -- consistent with the grounding goal even in the zero-cost, no-key configuration.

## Chat prompt (follow-up Q&A) — final text

In [ ]:
history = [{"role": "user", "content": "What's the overall trend?"}, {"role": "assistant", "content": "Bullish, per SMA50 > SMA200."}]
convo_lines = [f"{t['role']}: {t['content']}" for t in history]
convo_block = "\n".join(convo_lines)

chat_context_block = gc._serialize_context("AAPL", sample_metrics, None, None)

chat_prompt = (
    f"You are answering a follow-up question about ticker AAPL. Only use the DATA block below; "
    f"never invent numbers. If the answer requires data not present, say so explicitly.\n\n"
    f"DATA:\n{chat_context_block}\n\n"
    f"Conversation so far:\n{convo_block}\n\n"
    f"User's new question: Is it overbought right now?\n"
    f"Answer concisely."
)
print(chat_prompt)

## Summary of hallucination mitigations applied

1. **No open-ended prompts** — every prompt embeds a concrete JSON data block; the model is never asked to "write about {ticker}" without data.
2. **Explicit "never invent" instruction repeated at both the system-instruction and per-call level.**
3. **Explicit missing-data instruction** — the model is told to say "not available" rather than fill gaps, matching the same `null`/`N/A` handling already used in the UI (`api/analytics.py` never invents a fundamental field either — it returns `None` and the frontend renders `N/A`).
4. **Disclaimer always appended in code, not left to the model** — `generate_research_note`, `chat`, and `explain_peer_comparison` all append `"Educational project. Not investment advice."` in Python after the model call returns, so this line is never dependent on the model choosing to include it.
5. **try/except around every Gemini call** — if the call fails or returns an empty response, the code falls back to the exact same rule-based template used when no API key is configured at all, so there's no code path where an error becomes a blank or broken response shown to the user.
6. **No key required to exercise this design** — because the fallback path uses the identical section structure and the identical underlying data, this project's core grounding *behavior* is fully testable and demonstrable at zero cost, without ever calling the live Gemini API.

Disclaimer: Educational project. Not investment advice.